In [13]:
from dotenv import load_dotenv
from agents import Agent,Runner,trace,GuardrailFunctionOutput,function_tool,input_guardrail,output_guardrail
from agents import set_default_openai_client,set_tracing_export_api_key
import os
from openai import AsyncOpenAI
import sendgrid as sg
from sendgrid import Email,Mail,To,Content
from pydantic import BaseModel

In [14]:
load_dotenv(override=True)

client = AsyncOpenAI(base_url='https://api.aihubmix.com/v1',api_key=os.getenv("AIHUBMIX_API_KEY"))
set_default_openai_client(client=client,use_for_tracing=False)
set_tracing_export_api_key(os.getenv("TRACE_KEY"))

In [15]:
ins1 = "你是一个专业推销邮件的写手，为SEAI公司推销你们的AI Agent框架，写作风格是专业。"
ins2 = "你是一个专业推销邮件的写手，为SEAI公司推销你们的AI Agent框架，写作风格是幽默风趣能让潜在客户engaging。"
ins3 = "你是一个专业推销邮件的写手，为SEAI公司推销你们的AI Agent框架，写作风格是简短不废话。"

writer1 = Agent(name="writer1",instructions=ins1,model = 'gemini-3.1-flash-lite')
writer2 = Agent(name="writer2",instructions=ins2,model = 'gemini-3.1-flash-lite')
writer3 = Agent(name="writer3",instructions=ins3,model = 'gemini-3.1-flash-lite')

In [16]:
tool1 = writer1.as_tool(tool_name="writer1",tool_description="用中文写一封推销邮件")
tool2 = writer2.as_tool(tool_name="writer2",tool_description="用中文写一封推销邮件")
tool3 = writer3.as_tool(tool_name="writer3",tool_description="用中文写一封推销邮件")

tools = [tool1,tool2,tool3]

In [17]:
subj_agent = Agent(name="subject_writer",instructions="为邮件写一个主题",model="gpt-4o-mini")
subj_tool = subj_agent.as_tool(tool_name="subject_writer",tool_description="为邮件写主题")
html_agent = Agent(name="html_formatter",instructions="将邮件正文转化为html格式，并添加一个可以直接回复的按钮",model="gpt-4o-mini")
html_tool = html_agent.as_tool(tool_name="html_formatter",tool_description="将邮件正文转化为html格式，并添加一个可以直接回复的按钮")

In [18]:
@function_tool
def send_mail(html_input:str,subject:str):
    "以html的格式以及输入的主题发送邮件"
    sg_client = sg.SendGridAPIClient(os.getenv("SENDGRID_API_KEY"))
    from_email = Email("ivy@solareclipseai.men")
    to_email = To("solareclipse0130@gmail.com")
    content = Content(mime_type="text/html",content=html_input)
    mail = Mail(from_email=from_email,to_emails=to_email,subject=subject,html_content=content).get()
    
    sg_client.client.mail.send.post(request_body = mail)

    return {"status":"success"}

In [19]:
class check_output(BaseModel):
    is_offensive :bool
    is_confidential:bool

op_guardrail = Agent(name="og_agent",instructions="你需要检查邮件的内容是否有攻击性语言和隐私信息",model="gpt-4o-mini",output_type=check_output)

@output_guardrail
async def op_check(ctx,agent,message):
    result = await Runner.run(op_guardrail,message,context=ctx.context)
    cond1 = result.final_output.is_offensive 
    cond2 = result.final_output.is_confidential

    return GuardrailFunctionOutput(output_info=result,tripwire_triggered=(cond1 or cond2))

In [20]:
sending_ins = """
首先，给这封邮件取一个主题，然后，将邮件的正文内容转换为HTML，然后将它发出。
"""
tools2 = [subj_tool,html_tool,send_mail]

sending_agent = Agent(name="sending_agent",instructions=sending_ins,model = "gpt-5.4-nano",tools = tools2,output_guardrails=[op_check])

handoffs = [sending_agent]

In [21]:
class CheckNameUrl(BaseModel):
    NameExist:bool
    UrlExist:bool
    name:str
    url:str

ig_agent = Agent(name="input_checker",instructions="你需要检查用户的输入中是否包含姓名和网址",model="gpt-4o-mini",output_type=CheckNameUrl)

@input_guardrail
async def ig(ctx,agent,message):
    result = await Runner.run(ig_agent,input=message,context=ctx.context)
    cond0 = result.final_output.NameExist
    cond1 = result.final_output.UrlExist

    return GuardrailFunctionOutput(output_info=result.final_output,tripwire_triggered=not(cond0 and cond1))

In [22]:
manager_ins = """
你是一位专业的销售经理，需要从三位推销邮件写手所写的邮件中挑出一份最好的，传递给sending_agent。
需要注意的是：等三位写手写完之后你再做评判，不要自己写邮件，只做评判，挑出最好的邮件body。
"""

manager_agent = Agent(name="sales_manager",
                      instructions=manager_ins,
                      tools=tools,
                      handoffs=handoffs,
                      input_guardrails=[ig],
                    #   output_guardrails=[op_check],
                      model="gpt-5.4-nano")

In [23]:
message = "请你以Ivy的名义写一封推销邮件，不要忘了加上公司的主页：solareclipseai.men"

with trace("完整测试1"):
    res =await Runner.run(manager_agent,input=message)

In [24]:
res.final_output

'邮件已发送成功 ✅'